In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

df= pd.read_csv(r"C:\Users\Bhadmus\Documents\Codeveda Project\Online_Sales_Analytics\Task 4\New_cleaned_online_retail.csv")




In [2]:
# 1. Prepare Customer-Level Data
df['Invoicedate'] = pd.to_datetime(df['Invoicedate'])
latest_date = df['Invoicedate'].max()

customer_stats = df.groupby('Customer_Id').agg({
    'Invoicedate': lambda x: (latest_date - x.max()).days,
    'Invoice': 'nunique',
    'Revenue': 'sum'
}).rename(columns={
    'Invoicedate': 'Recency',
    'Invoice': 'Frequency',
    'Revenue': 'Monetary'
})

In [3]:
 #2. Define Churn
customer_stats['Churn'] = (customer_stats['Recency'] > 90).astype(int)

# 3. Features & Target
X = customer_stats[['Frequency', 'Monetary']]
y = customer_stats['Churn']

# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
 #5. Feature Scaling (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------
# MODEL 1: Logistic Regression
# -------------------------------
log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)
log_pred = log_model.predict(X_test_scaled)

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, log_pred))
print(classification_report(y_test, log_pred))

Logistic Regression
Accuracy: 0.716589861751152
              precision    recall  f1-score   support

           0       0.78      0.79      0.79       569
           1       0.59      0.58      0.58       299

    accuracy                           0.72       868
   macro avg       0.69      0.68      0.68       868
weighted avg       0.71      0.72      0.72       868



In [5]:
# -------------------------------
# MODEL 2: Decision Tree
# -------------------------------
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)

print("\nDecision Tree")
print("Accuracy:", accuracy_score(y_test, tree_pred))
print(classification_report(y_test, tree_pred))



Decision Tree
Accuracy: 0.6405529953917051
              precision    recall  f1-score   support

           0       0.71      0.76      0.74       569
           1       0.47      0.41      0.44       299

    accuracy                           0.64       868
   macro avg       0.59      0.59      0.59       868
weighted avg       0.63      0.64      0.63       868



In [6]:
#-------------------------------
# MODEL 3: Random Forest
# -------------------------------
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("\nRandom Forest")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))


Random Forest
Accuracy: 0.6486175115207373
              precision    recall  f1-score   support

           0       0.71      0.77      0.74       569
           1       0.49      0.41      0.45       299

    accuracy                           0.65       868
   macro avg       0.60      0.59      0.59       868
weighted avg       0.64      0.65      0.64       868



In [7]:
# -------------------------------
# HYPERPARAMETER TUNING (🔥 key)
# -------------------------------
params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), params, cv=3)
grid.fit(X_train, y_train)

print("\nBest Parameters:", grid.best_params_)


Best Parameters: {'max_depth': 10, 'n_estimators': 200}


In [8]:
# 1. Use the winner (Logistic Regression) to make predictions
# We use the scaled features because Logistic Regression needs them!
customer_stats['Predicted_Churn'] = log_model.predict(scaler.transform(X)) 

# 2. Save it to the CSV
customer_stats.to_csv('Final_Churn_Predictions.csv')